# 01 — Adaptive RAG Introduction

**Retrieval-Augmented Generation (RAG)** combines information retrieval with text generation. Instead of relying solely on an LLM's internal knowledge, RAG fetches relevant external information *at query time* and feeds it into the generation step.

**Adaptive RAG** takes this further: it *classifies* each incoming query and selects the best retrieval strategy automatically — document search, web search, a hybrid, or even just the LLM itself.

## What you will learn
- Core RAG concepts & the motivation for being adaptive
- High-level architecture of the Adaptive RAG pipeline
- A simple query‑classification demo with `QueryClassifier`

## 1. Why Adaptive RAG?

| Challenge | Solution in Adaptive RAG |
|-----------|--------------------------|
| **Factual questions** (e.g. "What is the capital of France?") | Document retrieval from a curated knowledge base |
| **Time‑sensitive questions** (e.g. "Latest news on AI regulation") | Web search for fresh information |
| **Complex / multi‑step questions** | Hybrid (documents + web) or Self‑RAG with reflection |
| **Simple factual questions the LLM already knows** | Direct LLM answer — no retrieval overhead |

The key insight: *one retrieval strategy does not fit all queries*. By classifying first, we save cost, reduce latency, and improve accuracy.

## 2. Architecture Overview

```
User Query
    │
    ▼
┌──────────────┐
│  QueryClassifier │  ← determines type, complexity, what sources are needed
└──────┬───────┘
       │
       ▼
┌──────────────┐
│ StrategySelector │  ← maps classification → strategy name
└──────┬───────┘
       │
       ▼
┌──────────────────────────────────┐
│           Strategy               │
│  ┌───────────┬──────────┬────┐  │
│  │DocumentRAG│ WebSearch │Self│  │ …
│  │ Hybrid    │ DirectLLM │    │  │
│  └───────────┴──────────┴────┘  │
└──────────────────────────────────┘
       │
       ▼
  Answer + Sources
```

The pipeline is implemented as a **stateful LangGraph** workflow in production, but the core logic is simple enough to explore piece‑by‑piece in these notebooks.

## 3. Import the Classifier

Let's import `QueryClassifier` from the actual backend. If the import fails (e.g. you haven't installed the package), a fallback class is provided.

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.router.query_classifier import QueryClassifier
    print("✓ Imported QueryClassifier from backend")
except ImportError as e:
    print(f"⚠ Could not import from backend: {e}")
    print("→ Using inline fallback class for demonstration\n")
    
    from dataclasses import dataclass
    @dataclass
    class ClassificationResult:
        query_type: str
        complexity: str
        needs_docs: bool
        needs_web: bool
        is_time_sensitive: bool
        needs_graph: bool
    
    class QueryClassifier:
        keywords_factual = {"what", "when", "where", "who", "which", "how many", "define"}
        keywords_exploratory = {"why", "explain", "describe", "tell me about"}
        keywords_comparative = {"compare", "versus", "vs", "difference", "different", "better", "worse"}
        keywords_procedural = {"how", "steps", "instructions", "guide", "process"}
        keywords_opinion = {"think", "believe", "should", "recommend", "best", "worst"}
        keywords_temporal = {"today", "yesterday", "latest", "recent", "now", "current", "breaking", "news"}
        keywords_graph = {"relationship", "connected", "associated", "entity", "entities"}
        
        def classify(self, query: str) -> ClassificationResult:
            q = query.lower()
            return ClassificationResult(
                query_type=self._determine_type(q),
                complexity=self._determine_complexity(q),
                needs_docs=not any(k in q for k in self.keywords_temporal),
                needs_web=any(k in q for k in self.keywords_temporal),
                is_time_sensitive=any(k in q for k in self.keywords_temporal),
                needs_graph=any(k in q for k in self.keywords_graph),
            )
        
        def _determine_type(self, q: str) -> str:
            if any(k in q for k in self.keywords_comparative): return "comparative"
            if any(k in q for k in self.keywords_procedural): return "procedural"
            if any(k in q for k in self.keywords_exploratory): return "exploratory"
            if any(k in q for k in self.keywords_opinion): return "opinion"
            return "factual"
        
        def _determine_complexity(self, q: str) -> str:
            words = len(q.split())
            conj = sum(1 for w in q.split() if w in {"and", "or", "but"})
            if words > 20 or conj >= 2: return "complex"
            if words > 10 or conj == 1: return "moderate"
            return "simple"

classifier = QueryClassifier()
print("✓ Ready to classify queries!")

## 4. Classify Some Queries

Run the cell below to see how the classifier labels different kinds of questions.

In [ ]:
queries = [
    "What is the capital of France?",
    "Why do humans dream?",
    "Compare Python vs JavaScript for web development",
    "How do I install PostgreSQL on macOS?",
    "What is the best laptop for programming?",
    "What are the latest developments in AI regulation?",
    "Show me the relationship between inflation and unemployment",
    "Define quantum entanglement and explain its implications for computing using multiple complex references spanning several domains",
]

print(f"{'Query':<65} {'Type':<14} {'Complexity':<12} {'Docs':<6} {'Web':<6} {'Time':<6}")
print("─" * 115)
for q in queries:
    r = classifier.classify(q)
    print(f"{q[:64]:<65} {r.query_type:<14} {r.complexity:<12} {r.needs_docs!s:<6} {r.needs_web!s:<6} {r.is_time_sensitive!s:<6}")

## 5. Key Takeaways

- **Query classification** is the *first* step in the adaptive pipeline.
- The classifier looks at keywords, sentence length, and conjunctions to produce a rich profile (`query_type`, `complexity`, `needs_docs`, `needs_web`, etc.).
- This profile drives strategy selection — which we'll explore in **Notebook 03**.

> **Next:** [02 — Query Classification](./02_Query_Classification.ipynb) — a deep dive into the classifier